# Clase 14 · Eliminación de variables con baja varianza: práctica guiada

Primer método del módulo para **quitar dimensiones**: borrar las columnas que **casi no varían**.

Usamos el **mismo dataset sintético de móviles de la clase 13** (20 variables, 4 rangos de precio, semilla 13) y la **misma división** (`stratify=y`, `random_state=1`), para comparar contra tu baseline.

Recuerda: en este dataset inventado **solo 4 columnas influyen en el precio** (`ram`, `battery_power`, `px_height`, `px_width`); las otras 16 son ruido. Eso nos deja comprobar si la varianza "acierta" o no.

> No hace falta matplotlib: las "gráficas" se muestran como tablas y barras de texto.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler


def generar_moviles(n=2000, semilla=13):
    """Versión sintética de 'mobile price classification' (la misma de la clase 13 y de los tests)."""
    rng = np.random.default_rng(semilla)
    d = pd.DataFrame({
        "battery_power": rng.integers(500, 2000, n), "blue": rng.integers(0, 2, n),
        "clock_speed": rng.uniform(0.5, 3.0, n).round(1), "dual_sim": rng.integers(0, 2, n),
        "fc": rng.integers(0, 20, n), "four_g": rng.integers(0, 2, n),
        "int_memory": rng.integers(2, 65, n), "m_dep": rng.uniform(0.1, 1.0, n).round(1),
        "mobile_wt": rng.integers(80, 201, n), "n_cores": rng.integers(1, 9, n),
        "pc": rng.integers(0, 21, n), "px_height": rng.integers(0, 1961, n),
        "px_width": rng.integers(500, 2000, n), "ram": rng.integers(256, 4000, n),
        "sc_h": rng.integers(5, 20, n), "sc_w": rng.integers(0, 19, n),
        "talk_time": rng.integers(2, 21, n), "three_g": rng.integers(0, 2, n),
        "touch_screen": rng.integers(0, 2, n), "wifi": rng.integers(0, 2, n),
    })
    z = lambda c: (d[c] - d[c].mean()) / d[c].std()
    puntaje = 3 * z("ram") + z("battery_power") + 0.7 * z("px_height") + 0.7 * z("px_width") + rng.normal(0, 0.5, n)
    d["price_range"] = pd.qcut(puntaje, 4, labels=False).astype(int)
    return d


data = generar_moviles()
X = data.drop(columns="price_range")
y = data["price_range"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)
print(X_train.shape, X_test.shape)

## 1. ¿Qué es la varianza? La media sola no basta
El ejemplo del profesor: un grupo A con valores **-5 y 5** y un grupo B con **-10 y 10**. Los dos tienen **media 0**, pero B está más disperso.

$$\text{varianza} = \frac{\sum (x - \bar{x})^2}{n}$$

In [ ]:
A = np.array([-5, 5])
B = np.array([-10, 10])
print("media A:", A.mean(), "| media B:", B.mean())

# varianza "a mano": diferencias respecto a la media, al cuadrado, sumadas y divididas entre n
var_A = ((A - A.mean()) ** 2).sum() / len(A)
var_B = ((B - B.mean()) ** 2).sum() / len(B)
print("varianza A:", var_A, "| varianza B:", var_B)
print("desviación estándar (raíz de la varianza) A:", np.sqrt(var_A), "| B:", np.sqrt(var_B))

La pregunta de Eduardo: ¿por qué varianza y no desviación estándar? La desviación estándar es la **raíz** de la varianza: da valores más "acotados" (5 y 10 en vez de 25 y 100). 📌 Extra: para **ordenar** columnas da igual; si una tiene más varianza, también tiene más desviación. Lo único que cambia es el número que pones como umbral.

## 2. El ejemplo de la pizarra
Cuatro muestras, tres variables y dos clases (0 y 1).

In [ ]:
pizarra = pd.DataFrame({"x1": [2, 10, 3, 15], "x2": [1, 2, 1, 1],
                        "x3": [100, 50, 130, 45], "y": [0, 1, 0, 1]})
print(pizarra)
print()
print("Varianza (ddof=0) de cada x:")
print(pizarra[["x1", "x2", "x3"]].var(ddof=0).sort_values(ascending=False))
print()
print("Mediana de cada x por clase (lo que mostraría el boxplot por clase):")
print(pizarra.groupby("y")[["x1", "x2", "x3"]].median())

- `x1` y `x3`: cada clase vive en un rango distinto → varianza alta.
- `x2`: casi no cambia → varianza baja → variable **débil**.

⚠️ 📌 Extra: el **boxplot por clase** sí mira la relación con `y`. `VarianceThreshold` **no**: solo calcula la varianza de la columna completa, sin saber qué clase es cada fila.

## 3. Ranking de varianza en los móviles

In [ ]:
ranking = X_train.var(ddof=0).sort_values(ascending=False)
ranking.round(4)

Fíjate: las 6 binarias (`blue`, `wifi`, …) están todas en **≈ 0.25**. 📌 Extra: una columna 0/1 **nunca** puede superar 0.25 (el máximo se da con mitad ceros y mitad unos: 0.5 × 0.5).

## 4. `VarianceThreshold` con umbral 0.5 (como en la clase)
Se ajusta **solo con `X_train`**. Conserva las columnas con varianza **estrictamente mayor** que el umbral.

In [ ]:
vt = VarianceThreshold(threshold=0.5)
X_train_vt = vt.fit_transform(X_train)     # fit + transform en train
X_test_vt = vt.transform(X_test)           # SOLO transform en test

print("Antes:", X_train.shape, "-> después:", X_train_vt.shape)
conservadas = list(X_train.columns[vt.get_support()])
print("Conservadas:", conservadas)
print("Eliminadas: ", list(X_train.columns[~vt.get_support()]))

De **20 a 13** variables, igual que en la clase, y con los mismos nombres que leyó el profesor: `battery_power`, `clock_speed`, `fc`, `int_memory`, …

Ojo: `fit_transform` devuelve un **array de numpy** (se pierden los nombres). Por eso usamos `get_support()` para recuperarlos.

## 5. ¿Qué umbral elijo? La curva "número de variables vs umbral"
El truco del profesor: probar umbrales de 0 a 2 de 0.1 en 0.1 y contar cuántas variables sobreviven.

In [ ]:
filas = []
for u in np.round(np.arange(0, 2.01, 0.1), 1):
    n = VarianceThreshold(threshold=u).fit(X_train).get_support().sum()
    filas.append({"umbral": u, "n_variables": int(n)})
curva = pd.DataFrame(filas)

# "gráfica" de texto: una barra por umbral
for _, f in curva.iterrows():
    print(f"{f['umbral']:>4.1f} | {'#' * int(f['n_variables'])} {int(f['n_variables'])}")

Se parece al **método del codo**: baja rápido (20 → 19 → 13 → 12) y luego se queda **plano en 12**. En la clase: 0 → 20, 0.25 → 18, 0.5 → 13, ≥ 0.75 → 12. En nuestros datos el escalón grande está entre 0.2 y 0.3: ahí caen las 6 binarias juntas.

## 6. ¿Mejora el modelo? (mismo Random Forest de la clase 13)

In [ ]:
def evaluar(Xtr, Xte):
    modelo = RandomForestClassifier(max_depth=5, max_leaf_nodes=20, random_state=1)
    modelo.fit(Xtr, y_train)
    return accuracy_score(y_test, modelo.predict(Xte))

acc_20 = evaluar(X_train, X_test)
acc_13 = evaluar(X_train[conservadas], X_test[conservadas])
print(f"R^20 (todas):           {acc_20:.4f}")
print(f"R^13 (varianza > 0.5):  {acc_13:.4f}")

Como en la clase (0.83 → 0.8475), el modelo con menos variables es **un poco mejor** (aquí 0.7525 → 0.76). La mejora es pequeña porque **todavía quedan 9 columnas de ruido** dentro de las 13 (`mobile_wt`, `int_memory`, `fc`, …). Lo vemos en la siguiente sección.

## 7. 📌 Extra · Tres cuidados con este método
### 7.1 La varianza depende de la ESCALA (las unidades)

In [ ]:
print("Varianza de ram en MB:", round(X_train["ram"].var(ddof=0), 1))
print("Varianza de ram en GB:", round((X_train["ram"] / 1024).var(ddof=0), 4))
print("¡Misma información, varianza un millón de veces menor!")
print()

esc = pd.DataFrame({
    "original": X_train.var(ddof=0),
    "minmax": np.var(MinMaxScaler().fit_transform(X_train), axis=0),
    "estandar": np.var(StandardScaler().fit_transform(X_train), axis=0),
}, index=X_train.columns)
esc.loc[["ram", "battery_power", "mobile_wt", "clock_speed", "blue", "wifi", "m_dep"]].round(4)

- **Original**: ram gana por estar en miles. Las binarias (máximo 0.25) **siempre** caen con un umbral ≥ 0.25, sean útiles o no.
- **MinMax** (todo en [0, 1]): el ranking se **invierte**; ahora las binarias tienen más varianza que ram.
- **Estandarizado**: todas valen **1** → `VarianceThreshold` ya no distingue nada.

Conclusión: el umbral solo tiene sentido si sabes **en qué escala** están tus columnas.

### 7.2 Varianza alta ≠ variable útil
`VarianceThreshold` es **no supervisado**: nunca mira `y`. Hagamos dos columnas inventadas:
- `binaria_util`: 1 si el precio es alto (2 o 3). ¡Separa las clases en dos grupos perfectos! Varianza 0.25.
- `ruido_gigante`: números al azar en miles. No dice nada del precio. Varianza enorme.

In [ ]:
rng = np.random.default_rng(0)
X_demo = X_train.copy()
X_demo["binaria_util"] = (y_train >= 2).astype(int)
X_demo["ruido_gigante"] = rng.normal(0, 1000, len(X_demo))

vt_demo = VarianceThreshold(0.5).fit(X_demo)
sobrevive = dict(zip(X_demo.columns, vt_demo.get_support()))
print("varianza binaria_util :", round(X_demo["binaria_util"].var(ddof=0), 4), "-> sobrevive:", sobrevive["binaria_util"])
print("varianza ruido_gigante:", round(X_demo["ruido_gigante"].var(ddof=0), 1), "-> sobrevive:", sobrevive["ruido_gigante"])
print()
print("Media por clase (¿separa los rangos de precio?):")
print(X_demo.assign(y=y_train).groupby("y")[["binaria_util", "ruido_gigante", "mobile_wt", "ram"]].mean().round(2))

El método borra la columna que separa las clases y conserva el ruido. También conserva `mobile_wt` (ruido de verdad en este dataset, varianza ≈ 1200). Para medir si una variable **ayuda a predecir `y`** hacen falta los métodos de las próximas clases.

### 7.3 `fit` en train, `transform` en test
Si ajustas el selector por separado en test, puede elegir **otras columnas**, y el modelo entrenado ya no encaja. Con un umbral cercano a 0.25 se ve claro:

In [ ]:
u = 0.2497
cols_train = set(X_train.columns[VarianceThreshold(u).fit(X_train).get_support()])
cols_test = set(X_test.columns[VarianceThreshold(u).fit(X_test).get_support()])
print("fit en train ->", len(cols_train), "columnas | fit en test ->", len(cols_test), "columnas")
print("Columnas en las que NO coinciden:", sorted(cols_train ^ cols_test))

Por eso: **`fit` (o `fit_transform`) solo con train** y **`transform` con test**. Así ambos quedan con las mismas columnas, decididas sin mirar el test.

## ✏️ Tu turno 1
Calcula **a mano** (sin `.var()` ni `np.var`) la varianza de `x1 = [2, 10, 3, 15]`. Debe darte **28.25**. Luego con `ddof=1` (dividir entre n − 1): ¿cuánto da?

In [ ]:
x1 = np.array([2, 10, 3, 15])
# tu código aquí


## ✏️ Tu turno 2
Repite la sección 6 con **umbral 1.0** y con **umbral 100**. ¿Cuántas columnas quedan en cada caso y qué accuracy obtienes? ¿Con umbral 100 queda alguna columna de ruido?

In [ ]:
# tu código aquí


## ✏️ Tu turno 3
Aplica `VarianceThreshold(0.5)` sobre `X_train` **estandarizado** con `StandardScaler`. ¿Cuántas columnas sobreviven? ¿Y con umbral 1.5? Explica por qué pasa eso.

In [ ]:
# tu código aquí
